# Кому VarLen semantic IDs помогают сильнее?

Гипотеза: variable-length semantic IDs дают больший прирост не всем пользователям, а прежде всего пользователям с длинной историей событий.

В этом notebook сравниваем paper-style setup: `fixed dVAE` vs `VarLen dVAE` на двух датасетах: **Yambda** и **Amazon**. Метрики считаются отдельно для пользователей с короткой, средней и длинной историей.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run this notebook from the repository root")

def run(cmd, *, check=True):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=ROOT, check=check)

PY = sys.executable
RESULTS_DIR = ROOT / "results" / "RQ_history_length"
RESULTS_DIR

## 1. Предусловия

Для **Yambda** нужны подготовленные файлы в `data/yambda/`: `dvae_train_items.parquet`, `dvae_holdout_items.parquet`, `seqrec_train_interactions.parquet`, `seqrec_test_sample_interactions.parquet` и т.д.

Для **Amazon** нужны подготовленные файлы в `data/amazon/`: `dvae_train_items.parquet`, `dvae_holdout_items.parquet`, `seqrec_train_interactions.parquet`, `seqrec_test_interactions.parquet` и т.д. Если их нет, сначала подготовьте датасеты через существующий data pipeline.

## 2. Yambda: fixed vs VarLen

Сначала строим train-holdout SIDs для fixed и VarLen dVAE, затем обучаем маленький seqrec. Если SIDs уже есть, первые две команды можно пропустить.

In [ ]:
run([PY, "-m", "scripts.train_dvae", "--config", "configs/RQ_history_length/yambda_fixed_dvae.yaml"])
run([PY, "-m", "scripts.train_dvae", "--config", "configs/RQ_history_length/yambda_varlen_dvae.yaml"])

run([PY, "-m", "scripts.train_seqrec", "--config", "configs/RQ_history_length/seqrec_yambda_fixed.yaml"])
run([PY, "-m", "scripts.train_seqrec", "--config", "configs/RQ_history_length/seqrec_yambda_varlen.yaml"])

## 3. Amazon: fixed vs VarLen

Для Amazon сначала строим train-holdout SIDs, затем обучаем такой же маленький seqrec. Перед seqrec создаем 10% user sample для test split. Если SIDs уже есть, первые две команды можно пропустить.

In [ ]:
run([PY, "-m", "scripts.train_dvae", "--config", "configs/RQ_history_length/amazon_fixed_dvae.yaml"])
run([PY, "-m", "scripts.train_dvae", "--config", "configs/RQ_history_length/amazon_varlen_dvae.yaml"])

run([
    PY, "-m", "scripts.RQ_history_length.build_test_sample",
    "--data-dir", "./data/amazon",
    "--user-fraction", "0.1",
])

run([PY, "-m", "scripts.train_seqrec", "--config", "configs/RQ_history_length/seqrec_amazon_fixed.yaml"])
run([PY, "-m", "scripts.train_seqrec", "--config", "configs/RQ_history_length/seqrec_amazon_varlen.yaml"])

## 4. Сбор результатов

Скрипт собирает `short / medium / long` бины для каждого датасета и считает дельту `varlen - fixed` внутри датасета.

In [ ]:
run([PY, "-m", "scripts.RQ_history_length.collect_history_results"])

comparison_path = RESULTS_DIR / "history_length_comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text())
    comparison
else:
    print("history_length_comparison.json is not created yet")

## 5. Как интерпретировать

Главное смотреть на строки `method = varlen` и поля:

- `delta_vs_fixed_recall@10`
- `delta_vs_fixed_recall@100`
- `delta_vs_fixed_ndcg@10`
- `delta_vs_fixed_ndcg@100`

Если дельта растет от `short` к `long` на Yambda и Amazon, это поддерживает гипотезу: VarLen semantic IDs особенно полезны пользователям с более длинной историей.

Формулировка вывода:

> В lightweight seqrec setup VarLen semantic IDs дают наибольший прирост относительно fixed-length dVAE для пользователей с [короткой / средней / длинной] историей. Эффект [согласован / не согласован] на Yambda и Amazon.